In [1]:
import numpy as np
from homicsx.mesh import import_mesh_auto
from mpi4py import MPI

# ------------------------------------------------------------------------------
# EXAMPLE 1: MESH FOLLOWING HOMICSX CONVENTION
# ------------------------------------------------------------------------------
import gmsh
import sys

def generate_homicsx_convention_mesh(
    filename: str = "rve_homicsx_convention.msh",
    Lx: float = 1.0,
    Ly: float = 1.0,
    inclusion_size: float = 0.3,
    mesh_size: float = 0.03,
):
    """
    Generate a 2D RVE mesh following HomiCSx PhysicalTags convention.
    
    Convention:
    - Matrix: physical group 1
    - Inclusion: physical group 10 (phase_id=0 + offset=10 → wait, matrix is phase 0)
    
    Actually, let's follow the exact convention:
    - Matrix (phase_id=0): physical group = PhysicalTags.matrix = 1
    - Inclusion (phase_id=1): physical group = PhysicalTags.phase_tag_offset + 1 = 10 + 1 = 11
    - Left boundary: physical group 3
    - Right boundary: physical group 4
    - Bottom boundary: physical group 5
    - Top boundary: physical group 6
    """
    gmsh.initialize()
    gmsh.model.add("homicsx_convention_rve")
    
    # --------------------------------------------------------------------------
    # Create matrix (outer square)
    # --------------------------------------------------------------------------
    matrix = gmsh.model.occ.addRectangle(0, 0, 0, Lx, Ly)
    
    # --------------------------------------------------------------------------
    # Create inclusion (inner square, centered)
    # --------------------------------------------------------------------------
    cx, cy = Lx / 2, Ly / 2
    half = inclusion_size / 2
    inclusion = gmsh.model.occ.addRectangle(
        cx - half, cy - half, 0, inclusion_size, inclusion_size
    )
    
    # --------------------------------------------------------------------------
    # Fragment: matrix minus inclusion
    # --------------------------------------------------------------------------
    all_parts, _ = gmsh.model.occ.fragment([(2, matrix)], [(2, inclusion)])
    gmsh.model.occ.synchronize()
    
    # --------------------------------------------------------------------------
    # Identify surfaces
    # --------------------------------------------------------------------------
    surfaces = gmsh.model.getEntities(dim=2)
    print(f"Number of surfaces: {len(surfaces)}")
    
    # Get surface areas to identify matrix vs inclusion
    matrix_surfaces = []
    inclusion_surfaces = []
    
    for surf in surfaces:
        # Get center of mass
        com = gmsh.model.occ.getCenterOfMass(surf[0], surf[1])
        area = gmsh.model.occ.getMass(surf[0], surf[1])
        
        # The inclusion is the smaller surface
        if abs(area - inclusion_size**2) < 1e-6:
            inclusion_surfaces.append(surf[1])
        else:
            matrix_surfaces.append(surf[1])
    
    print(f"Matrix surfaces: {matrix_surfaces}")
    print(f"Inclusion surfaces: {inclusion_surfaces}")
    
    # --------------------------------------------------------------------------
    # Assign physical groups following HomiCSx convention
    # --------------------------------------------------------------------------
    # Matrix → physical group 1
    gmsh.model.addPhysicalGroup(2, matrix_surfaces, 1)
    gmsh.model.setPhysicalName(2, 1, "Matrix")
    
    # Inclusion → physical group 11 (phase_tag_offset 10 + phase_id 1)
    gmsh.model.addPhysicalGroup(2, inclusion_surfaces, 11)
    gmsh.model.setPhysicalName(2, 11, "Inclusion_Phase_1")
    
    # --------------------------------------------------------------------------
    # Identify and tag boundaries
    # --------------------------------------------------------------------------
    lines = gmsh.model.getEntities(dim=1)
    
    left_lines = []
    right_lines = []
    bottom_lines = []
    top_lines = []
    
    for line in lines:
        com = gmsh.model.occ.getCenterOfMass(line[0], line[1])
        
        # Left boundary (x ≈ 0)
        if abs(com[0]) < 1e-6:
            left_lines.append(line[1])
        # Right boundary (x ≈ Lx)
        elif abs(com[0] - Lx) < 1e-6:
            right_lines.append(line[1])
        # Bottom boundary (y ≈ 0)
        elif abs(com[1]) < 1e-6:
            bottom_lines.append(line[1])
        # Top boundary (y ≈ Ly)
        elif abs(com[1] - Ly) < 1e-6:
            top_lines.append(line[1])
    
    # Tag boundaries following HomiCSx convention
    if left_lines:
        gmsh.model.addPhysicalGroup(1, left_lines, 3)
        gmsh.model.setPhysicalName(1, 3, "Left")
    if right_lines:
        gmsh.model.addPhysicalGroup(1, right_lines, 4)
        gmsh.model.setPhysicalName(1, 4, "Right")
    if bottom_lines:
        gmsh.model.addPhysicalGroup(1, bottom_lines, 5)
        gmsh.model.setPhysicalName(1, 5, "Bottom")
    if top_lines:
        gmsh.model.addPhysicalGroup(1, top_lines, 6)
        gmsh.model.setPhysicalName(1, 6, "Top")
    
    print(f"Left boundaries: {left_lines}")
    print(f"Right boundaries: {right_lines}")
    print(f"Bottom boundaries: {bottom_lines}")
    print(f"Top boundaries: {top_lines}")
    
    # --------------------------------------------------------------------------
    # Set mesh size and generate
    # --------------------------------------------------------------------------
    gmsh.model.occ.synchronize()
    
    # Set mesh size on all points
    points = gmsh.model.getEntities(dim=0)
    for point in points:
        gmsh.model.mesh.setSize([point], mesh_size)
    
    gmsh.model.mesh.generate(2)
    
    # --------------------------------------------------------------------------
    # Save
    # --------------------------------------------------------------------------
    gmsh.write(filename)
    print(f"\nMesh saved to: {filename}")
    
    gmsh.finalize()
    
    return filename


# ------------------------------------------------------------------------------
# GENERATE THE MESH
# ------------------------------------------------------------------------------
print("=" * 60)
print("GENERATING HOMICSX-CONVENTION MESH")
print("=" * 60)

generate_homicsx_convention_mesh(
    filename="rve_auto_import.msh",
    Lx=1.0,
    Ly=1.0,
    inclusion_size=0.3,
    mesh_size=0.03,
)

# ------------------------------------------------------------------------------
# AUTO IMPORT USING HOMICSX
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("AUTO IMPORT (Following HomiCSx Convention)")
print("=" * 60)

mesh, ct, ft = import_mesh_auto(
    mesh_file="rve_auto_import.msh",
    dim=2,
    domain_size=(1.0, 1.0),
)

print(f"\nImport successful!")
print(f"Mesh: {mesh}")
print(f"Cell tags - unique values: {np.unique(ct.values)}")
print(f"Facet tags - unique values: {np.unique(ft.values)}")

# Verify the tags
expected_cell_tags = {1, 11}  # matrix=1, inclusion=11
expected_facet_tags = {3, 4, 5, 6}  # left, right, bottom, top

actual_cell_tags = set(np.unique(ct.values))
actual_facet_tags = set(np.unique(ft.values))

print(f"\nCell tags match expected: {actual_cell_tags == expected_cell_tags}")
print(f"  Expected: {expected_cell_tags}")
print(f"  Actual:   {actual_cell_tags}")
print(f"\nFacet tags match expected: {actual_facet_tags == expected_facet_tags}")
print(f"  Expected: {expected_facet_tags}")
print(f"  Actual:   {actual_facet_tags}")

GENERATING HOMICSX-CONVENTION MESH
Number of surfaces: 2ments - Adding holes                                                                                           
Matrix surfaces: [3]
Inclusion surfaces: [2]
Left boundaries: [2]
Right boundaries: [3]
Bottom boundaries: [1]
Top boundaries: [4]
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Line)
Info    : [ 30%] Meshing curve 3 (Line)
Info    : [ 40%] Meshing curve 4 (Line)
Info    : [ 60%] Meshing curve 5 (Line)
Info    : [ 70%] Meshing curve 6 (Line)
Info    : [ 80%] Meshing curve 7 (Line)
Info    : [ 90%] Meshing curve 8 (Line)
Info    : Done meshing 1D (Wall 0.000318584s, CPU 0.000346s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 2 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 3 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0225597s, CPU 0.022092s)
Info    : 1434 nodes 2914 elements
Info    : Writing 'rve_auto_import.msh'...

Mesh saved t

In [2]:
# ------------------------------------------------------------------------------
# EXAMPLE 2: CUSTOM CONVENTION MESH + IMPORT WITH MAPPING
# ------------------------------------------------------------------------------
import numpy as np
from homicsx.mesh.importer import import_mesh_with_mapping, MeshImportMapping
from mpi4py import MPI
import gmsh

def generate_custom_convention_mesh(
    filename: str = "rve_custom_convention.msh",
    Lx: float = 1.0,
    Ly: float = 1.0,
    inclusion_size: float = 0.3,
    mesh_size: float = 0.03,
):
    """
    Generate a 2D RVE mesh with ARBITRARY physical group numbers.
    
    This mesh deliberately does NOT follow HomiCSx convention:
    - Matrix: physical group 100 (not 1)
    - Inclusion: physical group 200 (not 11)
    - Left boundary: physical group 301 (not 3)
    - Right boundary: physical group 302 (not 4)
    - Bottom boundary: physical group 303 (not 5)
    - Top boundary: physical group 304 (not 6)
    """
    gmsh.initialize()
    gmsh.model.add("custom_convention_rve")
    
    # --------------------------------------------------------------------------
    # Create matrix (outer square)
    # --------------------------------------------------------------------------
    matrix = gmsh.model.occ.addRectangle(0, 0, 0, Lx, Ly)
    
    # --------------------------------------------------------------------------
    # Create inclusion (inner square, centered)
    # --------------------------------------------------------------------------
    cx, cy = Lx / 2, Ly / 2
    half = inclusion_size / 2
    inclusion = gmsh.model.occ.addRectangle(
        cx - half, cy - half, 0, inclusion_size, inclusion_size
    )
    
    # --------------------------------------------------------------------------
    # Fragment: matrix minus inclusion
    # --------------------------------------------------------------------------
    all_parts, _ = gmsh.model.occ.fragment([(2, matrix)], [(2, inclusion)])
    gmsh.model.occ.synchronize()
    
    # --------------------------------------------------------------------------
    # Identify surfaces by area
    # --------------------------------------------------------------------------
    surfaces = gmsh.model.getEntities(dim=2)
    print(f"Number of surfaces: {len(surfaces)}")
    
    matrix_surfaces = []
    inclusion_surfaces = []
    
    for surf in surfaces:
        area = gmsh.model.occ.getMass(surf[0], surf[1])
        if abs(area - inclusion_size**2) < 1e-6:
            inclusion_surfaces.append(surf[1])
        else:
            matrix_surfaces.append(surf[1])
    
    print(f"Matrix surfaces: {matrix_surfaces}")
    print(f"Inclusion surfaces: {inclusion_surfaces}")
    
    # --------------------------------------------------------------------------
    # Assign CUSTOM physical groups (NOT HomiCSx convention!)
    # --------------------------------------------------------------------------
    gmsh.model.addPhysicalGroup(2, matrix_surfaces, 100)
    gmsh.model.setPhysicalName(2, 100, "Custom_Matrix")
    
    gmsh.model.addPhysicalGroup(2, inclusion_surfaces, 200)
    gmsh.model.setPhysicalName(2, 200, "Custom_Inclusion")
    
    # --------------------------------------------------------------------------
    # Identify and tag boundaries with CUSTOM tags
    # --------------------------------------------------------------------------
    lines = gmsh.model.getEntities(dim=1)
    
    left_lines = []
    right_lines = []
    bottom_lines = []
    top_lines = []
    
    for line in lines:
        com = gmsh.model.occ.getCenterOfMass(line[0], line[1])
        
        if abs(com[0]) < 1e-6:
            left_lines.append(line[1])
        elif abs(com[0] - Lx) < 1e-6:
            right_lines.append(line[1])
        elif abs(com[1]) < 1e-6:
            bottom_lines.append(line[1])
        elif abs(com[1] - Ly) < 1e-6:
            top_lines.append(line[1])
    
    # Tag with CUSTOM numbers
    if left_lines:
        gmsh.model.addPhysicalGroup(1, left_lines, 301)
        gmsh.model.setPhysicalName(1, 301, "Custom_Left")
    if right_lines:
        gmsh.model.addPhysicalGroup(1, right_lines, 302)
        gmsh.model.setPhysicalName(1, 302, "Custom_Right")
    if bottom_lines:
        gmsh.model.addPhysicalGroup(1, bottom_lines, 303)
        gmsh.model.setPhysicalName(1, 303, "Custom_Bottom")
    if top_lines:
        gmsh.model.addPhysicalGroup(1, top_lines, 304)
        gmsh.model.setPhysicalName(1, 304, "Custom_Top")
    
    print(f"Left: {left_lines}, Right: {right_lines}")
    print(f"Bottom: {bottom_lines}, Top: {top_lines}")
    
    # --------------------------------------------------------------------------
    # Generate mesh
    # --------------------------------------------------------------------------
    gmsh.model.occ.synchronize()
    
    points = gmsh.model.getEntities(dim=0)
    for point in points:
        gmsh.model.mesh.setSize([point], mesh_size)
    
    gmsh.model.mesh.generate(2)
    gmsh.write(filename)
    print(f"\nMesh saved to: {filename}")
    
    gmsh.finalize()
    return filename


# ------------------------------------------------------------------------------
# GENERATE THE CUSTOM CONVENTION MESH
# ------------------------------------------------------------------------------
print("=" * 60)
print("GENERATING CUSTOM-CONVENTION MESH")
print("=" * 60)

generate_custom_convention_mesh(
    filename="rve_custom_import.msh",
    Lx=1.0,
    Ly=1.0,
    inclusion_size=0.3,
    mesh_size=0.03,
)

# ------------------------------------------------------------------------------
# OPTION A: IMPORT WITH EXPLICIT BOUNDARY MAPPING
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("OPTION A: Import with explicit boundary mapping")
print("=" * 60)

mapping = MeshImportMapping(
    cell_groups={
        100: 1,   # Custom "Matrix" group 100 → HomiCSx matrix tag 1
        200: 11,  # Custom "Inclusion" group 200 → HomiCSx inclusion tag 11
    },
    boundary_groups={
        301: "left",    # Custom boundary 301 → left
        302: "right",   # Custom boundary 302 → right
        303: "bottom",  # Custom boundary 303 → bottom
        304: "top",     # Custom boundary 304 → top
    },
    auto_detect_boundaries=False,  # Use our explicit boundary mapping
)

mesh_a, ct_a, ft_a = import_mesh_with_mapping(
    mesh_file="rve_custom_import.msh",
    dim=2,
    domain_size=(1.0, 1.0),
    mapping=mapping,
)

print(f"\nImport successful!")
print(f"Cell tags - unique values: {np.unique(ct_a.values)}")
print(f"Facet tags - unique values: {np.unique(ft_a.values)}")

expected_cell_tags = {1, 11}
expected_facet_tags = {3, 4, 5, 6}

print(f"\nCell tags match HomiCSx: {set(np.unique(ct_a.values)) == expected_cell_tags}")
print(f"  Expected: {expected_cell_tags}")
print(f"  Actual:   {set(np.unique(ct_a.values))}")
print(f"\nFacet tags match HomiCSx: {set(np.unique(ft_a.values)) == expected_facet_tags}")
print(f"  Expected: {expected_facet_tags}")
print(f"  Actual:   {set(np.unique(ft_a.values))}")


# ------------------------------------------------------------------------------
# OPTION B: IMPORT WITH AUTO BOUNDARY DETECTION
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("OPTION B: Import with auto boundary detection")
print("=" * 60)

mapping_auto = MeshImportMapping(
    cell_groups={
        100: 1,   # Custom group 100 → matrix
        200: 11,  # Custom group 200 → inclusion
    },
    auto_detect_boundaries=True,  # <-- Auto-detect by coordinates
)

mesh_b, ct_b, ft_b = import_mesh_with_mapping(
    mesh_file="rve_custom_import.msh",
    dim=2,
    domain_size=(1.0, 1.0),
    mapping=mapping_auto,
)

print(f"\nImport successful!")
print(f"Cell tags - unique values: {np.unique(ct_b.values)}")
print(f"Facet tags - unique values: {np.unique(ft_b.values)}")

print(f"\nCell tags match HomiCSx: {set(np.unique(ct_b.values)) == expected_cell_tags}")
print(f"  Expected: {expected_cell_tags}")
print(f"  Actual:   {set(np.unique(ct_b.values))}")
print(f"\nFacet tags match HomiCSx: {set(np.unique(ft_b.values)) == expected_facet_tags}")
print(f"  Expected: {expected_facet_tags}")
print(f"  Actual:   {set(np.unique(ft_b.values))}")

GENERATING CUSTOM-CONVENTION MESH
Number of surfaces: 2ments - Adding holes                                                                                           
Matrix surfaces: [3]
Inclusion surfaces: [2]
Left: [2], Right: [3]
Bottom: [1], Top: [4]
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Line)
Info    : [ 30%] Meshing curve 3 (Line)
Info    : [ 40%] Meshing curve 4 (Line)
Info    : [ 60%] Meshing curve 5 (Line)
Info    : [ 70%] Meshing curve 6 (Line)
Info    : [ 80%] Meshing curve 7 (Line)
Info    : [ 90%] Meshing curve 8 (Line)
Info    : Done meshing 1D (Wall 0.000314792s, CPU 0.000357s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 2 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 3 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0228552s, CPU 0.022338s)
Info    : 1434 nodes 2914 elements
Info    : Writing 'rve_custom_import.msh'...

Mesh saved to: rve_custom_import.msh
Info    : Done w

In [3]:
# ------------------------------------------------------------------------------
# EXAMPLE 3: 3D CUSTOM CONVENTION MESH + IMPORT WITH MAPPING
# ------------------------------------------------------------------------------
import numpy as np
from homicsx.mesh.importer import import_mesh_with_mapping, MeshImportMapping
from mpi4py import MPI
import gmsh

def generate_custom_convention_mesh_3d(
    filename: str = "rve_custom_3d.msh",
    Lx: float = 1.0,
    Ly: float = 1.0,
    Lz: float = 1.0,
    sphere_radius: float = 0.2,
    mesh_size: float = 0.08,
):
    """
    Generate a 3D RVE mesh with ARBITRARY physical group numbers.
    
    This mesh deliberately does NOT follow HomiCSx convention:
    - Matrix: physical group 500
    - Inclusion: physical group 600
    - Left: 701, Right: 702, Bottom: 703, Top: 704, Near: 705, Far: 706
    """
    gmsh.initialize()
    gmsh.model.add("custom_3d_rve")
    
    # --------------------------------------------------------------------------
    # Create matrix (cube)
    # --------------------------------------------------------------------------
    matrix = gmsh.model.occ.addBox(0, 0, 0, Lx, Ly, Lz)
    
    # --------------------------------------------------------------------------
    # Create inclusion (sphere, centered)
    # --------------------------------------------------------------------------
    cx, cy, cz = Lx / 2, Ly / 2, Lz / 2
    sphere = gmsh.model.occ.addSphere(cx, cy, cz, sphere_radius)
    
    # --------------------------------------------------------------------------
    # Fragment: matrix minus sphere
    # --------------------------------------------------------------------------
    all_parts, _ = gmsh.model.occ.fragment([(3, matrix)], [(3, sphere)])
    gmsh.model.occ.synchronize()
    
    # --------------------------------------------------------------------------
    # Identify volumes by mass
    # --------------------------------------------------------------------------
    volumes = gmsh.model.getEntities(dim=3)
    print(f"Number of volumes: {len(volumes)}")
    
    sphere_vol = (4.0/3.0) * np.pi * sphere_radius**3
    
    matrix_volumes = []
    inclusion_volumes = []
    
    for vol in volumes:
        vol_mass = gmsh.model.occ.getMass(vol[0], vol[1])
        if abs(vol_mass - sphere_vol) < 1e-4:
            inclusion_volumes.append(vol[1])
        else:
            matrix_volumes.append(vol[1])
    
    print(f"Matrix volumes: {matrix_volumes}")
    print(f"Inclusion volumes: {inclusion_volumes}")
    
    # --------------------------------------------------------------------------
    # Assign CUSTOM physical groups
    # --------------------------------------------------------------------------
    gmsh.model.addPhysicalGroup(3, matrix_volumes, 500)
    gmsh.model.setPhysicalName(3, 500, "Custom_Matrix_3D")
    
    gmsh.model.addPhysicalGroup(3, inclusion_volumes, 600)
    gmsh.model.setPhysicalName(3, 600, "Custom_Inclusion_3D")
    
    # --------------------------------------------------------------------------
    # Identify and tag boundaries with CUSTOM tags
    # --------------------------------------------------------------------------
    surfaces = gmsh.model.getEntities(dim=2)
    
    left_surfs = []
    right_surfs = []
    bottom_surfs = []
    top_surfs = []
    near_surfs = []
    far_surfs = []
    
    for surf in surfaces:
        com = gmsh.model.occ.getCenterOfMass(surf[0], surf[1])
        
        if abs(com[0]) < 1e-6:
            left_surfs.append(surf[1])
        elif abs(com[0] - Lx) < 1e-6:
            right_surfs.append(surf[1])
        elif abs(com[1]) < 1e-6:
            bottom_surfs.append(surf[1])
        elif abs(com[1] - Ly) < 1e-6:
            top_surfs.append(surf[1])
        elif abs(com[2]) < 1e-6:
            near_surfs.append(surf[1])
        elif abs(com[2] - Lz) < 1e-6:
            far_surfs.append(surf[1])
    
    if left_surfs:
        gmsh.model.addPhysicalGroup(2, left_surfs, 701)
        gmsh.model.setPhysicalName(2, 701, "Custom_Left")
    if right_surfs:
        gmsh.model.addPhysicalGroup(2, right_surfs, 702)
        gmsh.model.setPhysicalName(2, 702, "Custom_Right")
    if bottom_surfs:
        gmsh.model.addPhysicalGroup(2, bottom_surfs, 703)
        gmsh.model.setPhysicalName(2, 703, "Custom_Bottom")
    if top_surfs:
        gmsh.model.addPhysicalGroup(2, top_surfs, 704)
        gmsh.model.setPhysicalName(2, 704, "Custom_Top")
    if near_surfs:
        gmsh.model.addPhysicalGroup(2, near_surfs, 705)
        gmsh.model.setPhysicalName(2, 705, "Custom_Near")
    if far_surfs:
        gmsh.model.addPhysicalGroup(2, far_surfs, 706)
        gmsh.model.setPhysicalName(2, 706, "Custom_Far")
    
    print(f"Left: {len(left_surfs)}, Right: {len(right_surfs)}")
    print(f"Bottom: {len(bottom_surfs)}, Top: {len(top_surfs)}")
    print(f"Near: {len(near_surfs)}, Far: {len(far_surfs)}")
    
    # --------------------------------------------------------------------------
    # Generate mesh
    # --------------------------------------------------------------------------
    gmsh.model.occ.synchronize()
    
    points = gmsh.model.getEntities(dim=0)
    for point in points:
        gmsh.model.mesh.setSize([point], mesh_size)
    
    gmsh.model.mesh.generate(3)
    gmsh.write(filename)
    print(f"\nMesh saved to: {filename}")
    
    gmsh.finalize()
    return filename


# ------------------------------------------------------------------------------
# GENERATE THE 3D CUSTOM MESH
# ------------------------------------------------------------------------------
print("=" * 60)
print("GENERATING 3D CUSTOM-CONVENTION MESH")
print("=" * 60)

generate_custom_convention_mesh_3d(
    filename="rve_custom_3d.msh",
    Lx=1.0,
    Ly=1.0,
    Lz=1.0,
    sphere_radius=0.2,
    mesh_size=0.08,
)

# ------------------------------------------------------------------------------
# OPTION A: IMPORT WITH EXPLICIT BOUNDARY MAPPING
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("OPTION A: 3D import with explicit boundary mapping")
print("=" * 60)

mapping_3d = MeshImportMapping(
    cell_groups={
        500: 1,   # Custom "Matrix" group 500 → HomiCSx matrix tag 1
        600: 11,  # Custom "Inclusion" group 600 → HomiCSx inclusion tag 11
    },
    boundary_groups={
        701: "left",
        702: "right",
        703: "bottom",
        704: "top",
        705: "near",
        706: "far",
    },
    auto_detect_boundaries=False,
)

mesh_a, ct_a, ft_a = import_mesh_with_mapping(
    mesh_file="rve_custom_3d.msh",
    dim=3,
    domain_size=(1.0, 1.0, 1.0),
    mapping=mapping_3d,
)

print(f"\nImport successful!")
print(f"Cell tags - unique values: {np.unique(ct_a.values)}")
print(f"Facet tags - unique values: {np.unique(ft_a.values)}")

expected_cell_tags = {1, 11}
expected_facet_tags = {3, 4, 5, 6, 7, 8}

print(f"\nCell tags match HomiCSx: {set(np.unique(ct_a.values)) == expected_cell_tags}")
print(f"  Expected: {expected_cell_tags}")
print(f"  Actual:   {set(np.unique(ct_a.values))}")
print(f"\nFacet tags match HomiCSx: {set(np.unique(ft_a.values)) == expected_facet_tags}")
print(f"  Expected: {expected_facet_tags}")
print(f"  Actual:   {set(np.unique(ft_a.values))}")


# ------------------------------------------------------------------------------
# OPTION B: IMPORT WITH AUTO BOUNDARY DETECTION
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("OPTION B: 3D import with auto boundary detection")
print("=" * 60)

mapping_auto_3d = MeshImportMapping(
    cell_groups={
        500: 1,   # Custom group 500 → matrix
        600: 11,  # Custom group 600 → inclusion
    },
    auto_detect_boundaries=True,
)

mesh_b, ct_b, ft_b = import_mesh_with_mapping(
    mesh_file="rve_custom_3d.msh",
    dim=3,
    domain_size=(1.0, 1.0, 1.0),
    mapping=mapping_auto_3d,
)

print(f"\nImport successful!")
print(f"Cell tags - unique values: {np.unique(ct_b.values)}")
print(f"Facet tags - unique values: {np.unique(ft_b.values)}")

print(f"\nCell tags match HomiCSx: {set(np.unique(ct_b.values)) == expected_cell_tags}")
print(f"  Expected: {expected_cell_tags}")
print(f"  Actual:   {set(np.unique(ct_b.values))}")
print(f"\nFacet tags match HomiCSx: {set(np.unique(ft_b.values)) == expected_facet_tags}")
print(f"  Expected: {expected_facet_tags}")
print(f"  Actual:   {set(np.unique(ft_b.values))}")


# ------------------------------------------------------------------------------
# VERIFY THE MESH IS READY FOR HOMOGENIZATION
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("VERIFICATION: Mesh ready for homogenization?")
print("=" * 60)

# Check that we have exactly the right number of boundary tags
unique_facets = np.unique(ft_b.values)
required_3d_boundaries = {3, 4, 5, 6, 7, 8}
missing = required_3d_boundaries - set(unique_facets)
extra = set(unique_facets) - required_3d_boundaries

if not missing and not extra:
    print("All 6 boundaries present and correct!")
elif missing:
    print(f"Missing boundaries: {missing}")
elif extra:
    print(f"  Extra facet tags: {extra} (may be interior surfaces)")

# Check cell tags
unique_cells = np.unique(ct_b.values)
if 1 in unique_cells:
    print("Matrix phase (tag 1) present")
if 11 in unique_cells:
    print("Inclusion phase (tag 11) present")

GENERATING 3D CUSTOM-CONVENTION MESH
Number of volumes: 2                                                                                                                          
Matrix volumes: [3]
Inclusion volumes: [2]
Left: 1, Right: 1
Bottom: 1, Top: 1
Near: 1, Far: 1
Info    : Meshing 1D...
Info    : [ 10%] Meshing curve 14 (Circle)
Info    : [ 30%] Meshing curve 16 (Line)
Info    : [ 30%] Meshing curve 17 (Line)
Info    : [ 40%] Meshing curve 18 (Line)
Info    : [ 50%] Meshing curve 19 (Line)
Info    : [ 50%] Meshing curve 20 (Line)
Info    : [ 60%] Meshing curve 21 (Line)
Info    : [ 70%] Meshing curve 22 (Line)
Info    : [ 70%] Meshing curve 23 (Line)
Info    : [ 80%] Meshing curve 24 (Line)
Info    : [ 90%] Meshing curve 25 (Line)
Info    : [ 90%] Meshing curve 26 (Line)
Info    : [100%] Meshing curve 27 (Line)
Info    : Done meshing 1D (Wall 0.00048175s, CPU 0.000539s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 7 (Sphere, Frontal-Delaunay)
Info    : [ 20%] Mes